## Alchemy


In [1]:
# Jupyter cell: Initialize and optimize SQLite in FLY

# 1) Imports
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

from infrastructure.config import Config

In [1]:
# 2) Load application configuration
#    Adjust constructor as needed (e.g., passing path to config file or env vars)
config = Config()

NameError: name 'Config' is not defined

In [ ]:
# 3) Create the SQLAlchemy engine with thread-safe and future settings
engine = create_engine(
    config.database.connection_string,
    connect_args={"check_same_thread": False},  # allow multi-thread access
    future=True,
)

In [ ]:
# 4) Execute SQLite PRAGMAs for performance & integrity
with engine.connect() as conn:  # type: ignore
    conn.execute(text("PRAGMA wal_checkpoint"))  # type: ignore

    # ou, se preferir garantir que todo o WAL seja aplicado e truncado:
    conn.execute(text("PRAGMA wal_checkpoint(FULL)"))  # type: ignore
    conn.execute(text("PRAGMA wal_checkpoint(RESTART)"))  # type: ignore
    conn.execute(text("PRAGMA wal_checkpoint(TRUNCATE)"))  # type: ignore
    conn.execute(text("PRAGMA optimize"))  # type: ignore # Run internal optimizations
    conn.execute(text("PRAGMA journal_mode=WAL"))  # type: ignore # Write-Ahead Logging
    conn.execute(text("PRAGMA synchronous=FULL"))  # type: ignore # Full disk sync for safety
    conn.execute(text("PRAGMA foreign_keys=ON"))  # type: ignore # Enforce FK constraints
    conn.execute(text("PRAGMA temp_store=MEMORY"))  # type: ignore # RAM for temp tables
    conn.execute(text("PRAGMA cache_size=-65536"))  # type: ignore # 64 MB cache

In [ ]:
# 5) Create a session factory for ORM transactions
Session = sessionmaker(  # type: ignore
    bind=engine,  # type: ignore
    autoflush=True,
    expire_on_commit=True,
)

In [ ]:
# 6) (Optional) Inspect existing tables
with Session() as session:  # type: ignore
    result = session.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))  # type: ignore
    tables = [row[0] for row in result]
    print("⏺️ Tables in database:", tables)

## YFINANCE

In [33]:
import yfinance as yf

tickers = ["LREN3.SA", "VALE3.SA"]

# Histórico diário
df = yf.download(tickers, start="1960-01-01", interval="1d",)


C:\Users\faust\AppData\Local\Temp\ipykernel_17748\3791307439.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, start="1960-01-01", interval="1d",)
[*********************100%***********************]  2 of 2 completed


In [34]:
df

Price             Close                    High                     Low  \
Ticker         LREN3.SA   VALE3.SA     LREN3.SA   VALE3.SA     LREN3.SA   
Date                                                                      
2000-01-03          NaN   1.389175          NaN   1.406043          NaN   
2000-01-04  5443.398926   1.356099  8467.510075   1.378921  5443.398926   
2000-01-05  5443.398926   1.356099  5443.398926   1.356099  5443.398926   
2000-01-06  5443.398926   1.356099  5443.398926   1.389174  5443.398926   
2000-01-07  5443.398926   1.405712  5443.398926   1.412658  5443.398926   
...                 ...        ...          ...        ...          ...   
2025-09-15    16.219999  57.500000    16.260000  57.549999    15.950000   
2025-09-16    16.900000  57.700001    17.090000  58.060001    16.350000   
2025-09-17    17.020000  57.799999    17.540001  58.020000    16.790001   
2025-09-18    17.049999  57.689999    17.150000  58.080002    16.940001   
2025-09-19    16.920000  57.919998    17.180000  58.189999    16.870001   

Price                         Open                 Volume            
Ticker       VALE3.SA     LREN3.SA   VALE3.SA    LREN3.SA  VALE3.SA  
Date                                                                 
2000-01-03   1.389175          NaN   1.389175         NaN    585600  
2000-01-04   1.356099  8467.510075   1.375944         0.0    782400  
2000-01-05   1.339561  5443.398926   1.339561         0.0   1876800  
2000-01-06   1.356099  5443.398926   1.356099         0.0    792000  
2000-01-07   1.372636  5443.398926   1.372636         0.0   5347200  
...               ...          ...        ...         ...       ...  
2025-09-15  56.980000    15.950000  57.480000  14180200.0  12457500  
2025-09-16  57.189999    16.350000  57.770000  19329800.0  15618500  
2025-09-17  57.360001    17.010000  57.540001  31079600.0  23116700  
2025-09-18  57.400002    17.040001  57.840000   8975200.0  17700400  
2025-09-19  57.549999    17.139999  57.970001  15882500.0  25374600  

[6458 rows x 10 columns]

### Preço atual e variação intraday

In [9]:
info = yf.Tickers(" ".join(tickers))

for t in tickers:
    data = info.tickers[t].info
    print(
        t,
        "Preço:", data.get("regularMarketPrice"),
        "Variação:", data.get("regularMarketChange"),
        "Variação %:", data.get("regularMarketChangePercent"),
    )


LREN3.SA Preço: 16.92 Variação: -0.12999916 Variação %: -0.76245844
VALE3.SA Preço: 57.92 Variação: 0.22999954 Variação %: 0.39868182


### Dados fundamentais

In [10]:
data = {}

for t in tickers:
    tk = yf.Ticker(t)
    info = tk.info
    data[t] = {
        "Nome": info.get("longName"),
        "Setor": info.get("sector"),
        "Indústria": info.get("industry"),
        "Valor de Mercado": info.get("marketCap"),
        "P/L": info.get("trailingPE"),
        "Lucro por Ação (EPS)": info.get("trailingEps"),
        "Dividend Yield": info.get("dividendYield"),
        "Beta": info.get("beta"),
        "Moeda": info.get("currency"),
    }

import pandas as pd
df = pd.DataFrame(data).T
df


,Nome,Setor,Indústria,Valor de Mercado,P/L,Lucro por Ação (EPS),Dividend Yield,Beta,Moeda
LREN3.SA,Lojas Renner S.A.,Consumer Cyclical,Department Stores,16776969216,12.916031,1.31,5.19,0.593,BRL
VALE3.SA,Vale S.A.,Basic Materials,Other Industrial Metals & Mining,247247650816,8.938272,6.48,11.48,1.039,BRL


In [16]:

t = yf.Ticker("VAle3.SA")

# Retorna todos os campos de fundamentos
info = t.info

# Converter para DataFrame para melhor leitura
df = pd.DataFrame(info.items(), columns=["Campo", "Valor"])
df

,Campo,Valor
0,address1,Praia de Botafogo
1,address2,"186 offices 1101, 1701, and 1801 Botafogo"
2,city,Rio De Janeiro
3,state,RJ
4,zip,22250-145
...,...,...
158,earningsCallTimestampEnd,1754056800
159,isEarningsDateEstimate,False
160,epsTrailingTwelveMonths,6.48
161,epsForward,11.94


### Dados financeiros estruturados

In [18]:
import yfinance as yf

tickers = ["LREN3.SA"]

for t in tickers:
    tk = yf.Ticker(t)

    print(f"\n--- {t} ---")

    # Demonstrativos de resultado (anual e trimestral)
    print("Income Statement (anual):")
    print(tk.financials)

    print("\nIncome Statement (trimestral):")
    print(tk.quarterly_financials)

    # Balanço patrimonial
    print("\nBalance Sheet (anual):")
    print(tk.balance_sheet)

    print("\nBalance Sheet (trimestral):")
    print(tk.quarterly_balance_sheet)

    # Fluxo de caixa
    print("\nCashflow (anual):")
    print(tk.cashflow)

    print("\nCashflow (trimestral):")
    print(tk.quarterly_cashflow)



--- LREN3.SA ---
Income Statement (anual):
                                                      2024-12-31  \
Tax Effect Of Unusual Items                        -1.025383e+08   
Tax Rate For Calcs                                  9.410000e-02   
Normalized EBITDA                                   3.972445e+09   
Total Unusual Items                                -1.089674e+09   
Total Unusual Items Excluding Goodwill             -1.089674e+09   
Net Income From Continuing Operation Net Minori...  1.196668e+09   
Reconciled Depreciation                             1.244254e+09   
Reconciled Cost Of Revenue                          5.694417e+09   
EBITDA                                              2.882771e+09   
EBIT                                                1.638517e+09   
Net Interest Income                                -5.202990e+08   
Interest Expense                                    3.175200e+08   
Interest Income                                     3.124010e+08   
Norm

In [28]:
tk.eps_revisions

,upLast7days,upLast30days,downLast30days,downLast7Days
period,,,,
0q,0,0,2,0
+1q,0,1,1,0
0y,1,3,4,0
+1y,1,2,5,3
